# Real Yield - Notebook 2: Signal Construction


## 1. Load Aligned Panel


In [ ]:
# Cell 1: Drive mount (top of every notebook)
from google.colab import drive
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/macro_strategies/'
import os
os.makedirs(BASE_PATH + 'data/raw', exist_ok=True)
os.makedirs(BASE_PATH + 'data/processed', exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
from src.signal_builder import normalise_signal
panel = pd.read_parquet(BASE_PATH + 'data/processed/aligned_panel.parquet')


## 2. Raw Signal Computation

Implement the strategy's core formula (see `index.md`).


In [ ]:
# Real yield: 5Y nominal Treasury minus 5Y breakeven inflation (Fisher decomposition)
raw = panel['GS5'] - panel['T5YIE']

## 3. Signal Transformations

Rolling z-score, +/- 3 sigma clip, lag-1.


In [ ]:
signal = normalise_signal(raw, window=60)


## 4. Alternative Signal Variants

Headline vs. core, 3m vs. 6m, level vs. change.


In [ ]:
variants = pd.DataFrame({
    '5y_nominal_less_bei': normalise_signal(panel['GS5'] - panel['T5YIE']),
    'tips_5y_direct':      normalise_signal(panel['DFII5']) if 'DFII5' in panel.columns else pd.Series(dtype=float),
    '5y_bei_level':        normalise_signal(panel['T5YIE']),
}).dropna(axis=1, how='all')
variants.plot(figsize=(12, 5), title='Real Yield Signal Variants')

## 5. Signal Validation

Time-series plot, distribution histogram, autocorrelation.


In [ ]:
signal.plot(figsize=(12, 5), title='Signal (z-score)')


## 6. Predictive Power Pre-Backtest

Rolling Information Coefficient, quantile forward returns.


In [ ]:
import matplotlib.pyplot as plt

mkt_returns = pd.read_parquet(BASE_PATH + 'data/processed/market_returns.parquet')
fwd_returns = mkt_returns.iloc[:, 0].shift(-1)

aligned = pd.concat([signal, fwd_returns], axis=1).dropna()
aligned.columns = ['signal', 'fwd_ret']

ic = aligned['signal'].rolling(12).corr(aligned['fwd_ret'])
fig, ax = plt.subplots(figsize=(12, 4))
ic.plot(ax=ax, title='Rolling 12m Information Coefficient (vs primary asset fwd return)')
ax.axhline(0, color='black', linewidth=0.5, alpha=0.5)
ax.set_ylabel('IC (Pearson)')
plt.tight_layout()
plt.show()

aligned['quintile'] = pd.qcut(aligned['signal'], 5, labels=False, duplicates='drop')
qmeans = aligned.groupby('quintile')['fwd_ret'].mean() * 12
fig2, ax2 = plt.subplots(figsize=(8, 4))
qmeans.plot(kind='bar', ax=ax2, color='steelblue',
            title='Signal Quintile — Mean Annualised Forward Return')
ax2.set_xlabel('Quintile (1=low, 5=high)')
ax2.set_ylabel('Annualised Return')
plt.tight_layout()
plt.show()

## 7. Export


In [ ]:
signal.to_frame('signal').to_parquet(
    BASE_PATH + 'data/processed/signals.parquet'
)


## Limitations

- Rolling z-score is symmetric -- asymmetric distributions may bias.
- Lag of 1 assumes monthly release cadence.
